In [1]:
!pip install mlflow boto3 awscli optuna imbalanced-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# !aws configure  # skipped: interactive; not needed with local MLflow

In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("file:./mlruns")

In [4]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location=('file:///C:/Users/HAI/Downloads/Youtube sentiment '
 'analysis/notebooks/mlruns/775459616684597847'), creation_time=1787582478269, experiment_id='775459616684597847', last_update_time=1787582478269, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}>

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import RandomOverSampler
import mlflow
import mlflow.sklearn
import optuna


C:\Users\HAI\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import os
df = pd.read_csv(('tweets_preprocessing.csv' if os.path.exists('tweets_preprocessing.csv') else '/content/tweets_preprocessing.csv' if os.path.exists('/content/tweets_preprocessing.csv') else '../tweets_preprocessing.csv')).dropna()
df.shape

(54036, 14)

In [7]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Logistic Regression

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    # LogisticRegression model setup with balanced class weight
    model = LogisticRegression(C=C, penalty=penalty, solver='liblinear', random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for Logistic Regression, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LogisticRegression(C=best_params['C'], penalty=best_params['penalty'], solver='liblinear', random_state=42)

    # Log the best model with MLflow, passing the algo_name as "LogisticRegression"
    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Logistic Regression
run_optuna_experiment()


[I 2026-08-24 21:36:15,167] A new study created in memory with name: no-name-38875124-9ced-42b0-b8c2-fe3fefe17eb9


[I 2026-08-24 21:36:15,926] Trial 0 finished with value: 0.6679306608884074 and parameters: {'C': 0.010153571468071027, 'penalty': 'l2'}. Best is trial 0 with value: 0.6679306608884074.


[I 2026-08-24 21:36:17,637] Trial 1 finished with value: 0.752979414951246 and parameters: {'C': 2.741645995372458, 'penalty': 'l1'}. Best is trial 1 with value: 0.752979414951246.


[I 2026-08-24 21:36:19,477] Trial 2 finished with value: 0.7508899551153072 and parameters: {'C': 0.6623431149522205, 'penalty': 'l2'}. Best is trial 1 with value: 0.752979414951246.


[I 2026-08-24 21:36:20,418] Trial 3 finished with value: 0.7526698653459217 and parameters: {'C': 0.5311613877829957, 'penalty': 'l1'}. Best is trial 1 with value: 0.752979414951246.


...

[I 2026-08-24 21:36:55,447] Trial 26 finished with value: 0.7545271629778671 and parameters: {'C': 7.909438115856514, 'penalty': 'l2'}. Best is trial 12 with value: 0.7549140999845225.


[I 2026-08-24 21:36:56,941] Trial 27 finished with value: 0.7539080637672188 and parameters: {'C': 1.5238242247261564, 'penalty': 'l1'}. Best is trial 12 with value: 0.7549140999845225.


[I 2026-08-24 21:36:57,809] Trial 28 finished with value: 0.7526698653459217 and parameters: {'C': 0.5092772189119693, 'penalty': 'l1'}. Best is trial 12 with value: 0.7549140999845225.


[I 2026-08-24 21:36:59,220] Trial 29 finished with value: 0.7445441882061601 and parameters: {'C': 0.2306692164923359, 'penalty': 'l2'}. Best is trial 12 with value: 0.7549140999845225.


2026/08/24 21:37:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
